# Q-Regime
## Small-Data Quantum Kernel Learning for Financial Volatility Regime Classification

This notebook contains the final reproducible pipeline for comparing
classical and quantum kernel methods for short-horizon S&P 500
volatility-regime classification.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    f1_score,
    recall_score,
)

from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Aicha L\Q-Regime


## 1. Load source data

In [2]:
# Load S&P 500 daily market data
START_DATE = "2010-01-01"
END_DATE = "2025-01-01"

sp500 = yf.download(
    "^GSPC",
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=False
)

# Flatten yfinance MultiIndex columns if needed
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)

sp500.index = pd.to_datetime(sp500.index)
sp500.index.name = "Date"

# Load Bloomberg Garman-Klass historical volatility
bloomberg_vol = pd.read_csv(
    PROCESSED_DIR / "spx_bloomberg_gk_volatility.csv",
    parse_dates=["Date"]
)

bloomberg_vol = bloomberg_vol.set_index("Date")

print("Yahoo shape:", sp500.shape)
print("Yahoo range:", sp500.index.min(), "to", sp500.index.max())

print("\nBloomberg shape:", bloomberg_vol.shape)
print("Bloomberg range:", bloomberg_vol.index.min(), "to", bloomberg_vol.index.max())

Yahoo shape: (3774, 6)
Yahoo range: 2010-01-04 00:00:00 to 2024-12-31 00:00:00

Bloomberg shape: (3774, 2)
Bloomberg range: 2010-01-04 00:00:00 to 2024-12-31 00:00:00


## 2. Merge market and volatility data

In [3]:
# Merge Yahoo market data with Bloomberg Garman-Klass volatility
data = sp500.join(
    bloomberg_vol,
    how="inner"
)

# Rename Bloomberg columns for cleaner code
data = data.rename(columns={
    "SPX Index Hist Vol (5)": "gk_vol_5d",
    "SPX Index Hist Vol (20)": "gk_vol_20d"
})

print("Merged shape:", data.shape)
print("Date range:", data.index.min(), "to", data.index.max())

print("\nMissing values:")
print(data.isna().sum())

data.head()

Merged shape: (3774, 8)
Date range: 2010-01-04 00:00:00 to 2024-12-31 00:00:00

Missing values:
Adj Close     0
Close         0
High          0
Low           0
Open          0
Volume        0
gk_vol_5d     0
gk_vol_20d    0
dtype: int64


,Adj Close,Close,High,Low,Open,Volume,gk_vol_5d,gk_vol_20d
Date,,,,,,,,
2010-01-04,1132.989990,1132.989990,1133.869995,1116.560059,1116.560059,3991400000,6.895809,8.975828
2010-01-05,1136.520020,1136.520020,1136.630005,1129.660034,1132.660034,2491020000,6.737916,7.516543
2010-01-06,1137.140015,1137.140015,1139.189941,1133.949951,1135.709961,4972660000,6.907034,7.266088
2010-01-07,1141.689941,1141.689941,1142.459961,1131.319946,1136.270020,5270680000,8.006896,7.198322
2010-01-08,1144.979980,1144.979980,1145.390015,1136.219971,1140.520020,4389590000,8.077482,7.032808


## 3. Create prediction target

In [4]:
# Future 5-day Garman-Klass volatility target
# At date t, use the Bloomberg 5-day historical volatility observed 5 trading days later.
data["future_gk_vol_5d"] = data["gk_vol_5d"].shift(-5)

print(
    data[
        ["gk_vol_5d", "future_gk_vol_5d"]
    ].tail(10)
)

            gk_vol_5d  future_gk_vol_5d
Date                                   
2024-12-17   5.124125         17.694100
2024-12-18  12.098636         14.167424
2024-12-19  12.943466         14.460431
2024-12-20  16.658055         11.182908
2024-12-23  17.583497         10.412080
2024-12-24  17.694100               NaN
2024-12-26  14.167424               NaN
2024-12-27  14.460431               NaN
2024-12-30  11.182908               NaN
2024-12-31  10.412080               NaN


In [5]:
H = 5

train_raw = data.loc["2010-01-01":"2020-12-31"].copy()
val_raw   = data.loc["2021-01-01":"2022-12-31"].copy()
test_raw  = data.loc["2023-01-01":"2024-12-31"].copy()

# Prevent future target windows from crossing split boundaries
train_raw = train_raw.iloc[:-H].copy()
val_raw = val_raw.iloc[:-H].copy()

print(
    "Train:",
    train_raw.index.min(),
    "to",
    train_raw.index.max(),
    train_raw.shape
)

print(
    "Validation:",
    val_raw.index.min(),
    "to",
    val_raw.index.max(),
    val_raw.shape
)

print(
    "Test:",
    test_raw.index.min(),
    "to",
    test_raw.index.max(),
    test_raw.shape
)

Train: 2010-01-04 00:00:00 to 2020-12-23 00:00:00 (2764, 9)
Validation: 2021-01-04 00:00:00 to 2022-12-22 00:00:00 (498, 9)
Test: 2023-01-03 00:00:00 to 2024-12-31 00:00:00 (502, 9)


## 4. Define high-volatility regime

In [6]:
high_vol_threshold = (
    train_raw["future_gk_vol_5d"]
    .dropna()
    .quantile(0.67)
)

print("High-volatility threshold:", high_vol_threshold)

High-volatility threshold: 11.2243938342


In [7]:
data["high_vol"] = (
    (data["future_gk_vol_5d"] > high_vol_threshold)
    .astype(float)
    .where(data["future_gk_vol_5d"].notna())
)

print(
    data[
        ["future_gk_vol_5d", "high_vol"]
    ].tail(10)
)

            future_gk_vol_5d  high_vol
Date                                  
2024-12-17         17.694100       1.0
2024-12-18         14.167424       1.0
2024-12-19         14.460431       1.0
2024-12-20         11.182908       0.0
2024-12-23         10.412080       0.0
2024-12-24               NaN       NaN
2024-12-26               NaN       NaN
2024-12-27               NaN       NaN
2024-12-30               NaN       NaN
2024-12-31               NaN       NaN


## 5. Create model features

In [8]:
# Daily log return
data["log_return"] = np.log(
    data["Close"] / data["Close"].shift(1)
)

# 1-day return
data["return_1d"] = data["log_return"]

# 5-day momentum
data["momentum_5d"] = (
    data["log_return"]
    .rolling(window=5)
    .sum()
)

feature_cols = [
    "return_1d",
    "momentum_5d",
    "gk_vol_5d",
    "gk_vol_20d"
]

model_data = data[
    feature_cols
    + ["future_gk_vol_5d", "high_vol"]
].dropna().copy()

print("Model data shape:", model_data.shape)
print("Date range:", model_data.index.min(), "to", model_data.index.max())

model_data.head()

Model data shape: (3764, 6)
Date range: 2010-01-11 00:00:00 to 2024-12-23 00:00:00


,return_1d,momentum_5d,gk_vol_5d,gk_vol_20d,future_gk_vol_5d,high_vol
Date,,,,,,
2010-01-11,0.001745,0.012272,7.661832,7.061430,10.355161,0.0
2010-01-12,-0.009425,-0.000264,8.449730,7.241153,11.928864,1.0
2010-01-13,0.008291,0.007482,10.081274,7.771405,13.199877,1.0
2010-01-14,0.002423,0.005912,9.414070,7.731066,14.483879,1.0
2010-01-15,-0.010882,-0.007847,10.363180,8.018520,14.080746,1.0


## 6. Create chronological train / validation / test splits

In [9]:
H = 5

train = model_data.loc["2010-01-01":"2020-12-31"].iloc[:-H].copy()
val   = model_data.loc["2021-01-01":"2022-12-31"].iloc[:-H].copy()
test  = model_data.loc["2023-01-01":"2024-12-31"].copy()

print(
    "Train:",
    train.shape,
    train.index.min(),
    "to",
    train.index.max()
)

print(
    "Validation:",
    val.shape,
    val.index.min(),
    "to",
    val.index.max()
)

print(
    "Test:",
    test.shape,
    test.index.min(),
    "to",
    test.index.max()
)

print("\nHigh-volatility rates:")
print("Train:", train["high_vol"].mean())
print("Validation:", val["high_vol"].mean())
print("Test:", test["high_vol"].mean())

Train: (2759, 6) 2010-01-11 00:00:00 to 2020-12-23 00:00:00
Validation: (498, 6) 2021-01-04 00:00:00 to 2022-12-22 00:00:00
Test: (497, 6) 2023-01-03 00:00:00 to 2024-12-23 00:00:00

High-volatility rates:
Train: 0.33055454874954693
Validation: 0.5783132530120482
Test: 0.2575452716297787


## 7. Define model inputs and persistence baseline

In [10]:
X_train = train[feature_cols]
y_train = train["high_vol"].astype(int)

X_val = val[feature_cols]
y_val = val["high_vol"].astype(int)

X_test = test[feature_cols]
y_test = test["high_vol"].astype(int)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (2759, 4)
X_val: (498, 4)
X_test: (497, 4)


In [11]:
persistence_auc_5d = roc_auc_score(
    y_val,
    X_val["gk_vol_5d"]
)

persistence_auc_20d = roc_auc_score(
    y_val,
    X_val["gk_vol_20d"]
)

print("5-day GK persistence AUC:", persistence_auc_5d)
print("20-day GK persistence AUC:", persistence_auc_20d)

5-day GK persistence AUC: 0.8591269841269841
20-day GK persistence AUC: 0.8520337301587302


## 8. Classical RBF-SVM baseline with quantum-matched preprocessing

In [12]:
# Scale all features to [0, pi]
scaler = MinMaxScaler(
    feature_range=(0, np.pi),
    clip=True
)

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Classical RBF-SVM
classical_model = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    class_weight="balanced",
    random_state=42
)

classical_model.fit(
    X_train_scaled,
    y_train
)

y_val_pred = classical_model.predict(X_val_scaled)
y_val_score = classical_model.decision_function(X_val_scaled)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_val, y_val_pred)
)

print(
    "ROC-AUC:",
    roc_auc_score(y_val, y_val_score)
)

print(
    "F1:",
    f1_score(y_val, y_val_pred)
)

print(
    "Recall High Vol:",
    recall_score(y_val, y_val_pred)
)

Balanced Accuracy: 0.7957341269841269
ROC-AUC: 0.8908234126984127
F1: 0.8424543946932007
Recall High Vol: 0.8819444444444444


## 9. Small-data chronological windows

In [13]:
H = 5

# Reduce overlap between neighboring 5-day targets
train_thin = train.iloc[::H].copy()
val_thin = val.iloc[::H].copy()

print("Original train:", len(train))
print("Thinned train:", len(train_thin))

print("Original validation:", len(val))
print("Thinned validation:", len(val_thin))

print("\nHigh-volatility rates:")
print("Train thin:", train_thin["high_vol"].mean())
print("Validation thin:", val_thin["high_vol"].mean())

Original train: 2759
Thinned train: 552
Original validation: 498
Thinned validation: 100

High-volatility rates:
Train thin: 0.3278985507246377
Validation thin: 0.59


In [14]:
N_values = [20, 50, 100, 200]
n_windows = 3

window_indices = {}

for N in N_values:

    max_start = len(train_thin) - N

    starts = np.linspace(
        0,
        max_start,
        n_windows,
        dtype=int
    )

    window_indices[N] = []

    print(f"\nN = {N}")

    for window_id, start in enumerate(starts, start=1):

        window = train_thin.iloc[start:start + N]

        window_indices[N].append(window.index)

        print(
            f"Window {window_id}:",
            window.index.min().date(),
            "to",
            window.index.max().date(),
            "| High-vol rate:",
            round(window["high_vol"].mean(), 3)
        )


N = 20
Window 1: 2010-01-11 to 2010-05-27 | High-vol rate: 0.55
Window 2: 2015-04-24 to 2015-09-09 | High-vol rate: 0.25
Window 3: 2020-08-05 to 2020-12-18 | High-vol rate: 0.5

N = 50
Window 1: 2010-01-11 to 2010-12-30 | High-vol rate: 0.6
Window 2: 2015-01-06 to 2015-12-24 | High-vol rate: 0.32
Window 3: 2019-12-31 to 2020-12-18 | High-vol rate: 0.62

N = 100
Window 1: 2010-01-11 to 2011-12-27 | High-vol rate: 0.6
Window 2: 2014-07-09 to 2016-06-24 | High-vol rate: 0.32
Window 3: 2019-01-03 to 2020-12-18 | High-vol rate: 0.41

N = 200
Window 1: 2010-01-11 to 2013-12-23 | High-vol rate: 0.41
Window 2: 2013-07-11 to 2017-06-22 | High-vol rate: 0.17
Window 3: 2017-01-06 to 2020-12-18 | High-vol rate: 0.325


## 10. Small-data classical RBF-SVM

In [15]:
classical_small_results = []

for N in N_values:

    for window_id, idx in enumerate(window_indices[N], start=1):

        window = train_thin.loc[idx]

        X_train_window = window[feature_cols]
        y_train_window = window["high_vol"].astype(int)

        X_val_window = val_thin[feature_cols]
        y_val_window = val_thin["high_vol"].astype(int)

        # Same scaling that will be used for the quantum models
        scaler = MinMaxScaler(
            feature_range=(0, np.pi),
            clip=True
        )

        X_train_scaled = scaler.fit_transform(X_train_window)
        X_val_scaled = scaler.transform(X_val_window)

        model = SVC(
            kernel="rbf",
            C=1,
            gamma="scale",
            class_weight="balanced",
            random_state=42
        )

        model.fit(X_train_scaled, y_train_window)

        y_pred = model.predict(X_val_scaled)
        y_score = model.decision_function(X_val_scaled)

        classical_small_results.append({
            "N": N,
            "window": window_id,
            "start": window.index.min(),
            "end": window.index.max(),
            "train_high_vol_rate": y_train_window.mean(),
            "balanced_accuracy": balanced_accuracy_score(
                y_val_window, y_pred
            ),
            "roc_auc": roc_auc_score(
                y_val_window, y_score
            ),
            "f1": f1_score(
                y_val_window, y_pred
            ),
            "recall_high_vol": recall_score(
                y_val_window, y_pred
            )
        })

        print(f"Finished N={N}, window={window_id}")

classical_small_results_df = pd.DataFrame(
    classical_small_results
)

classical_small_results_df

Finished N=20, window=1
Finished N=20, window=2
Finished N=20, window=3
Finished N=50, window=1
Finished N=50, window=2
Finished N=50, window=3
Finished N=100, window=1
Finished N=100, window=2
Finished N=100, window=3
Finished N=200, window=1
Finished N=200, window=2
Finished N=200, window=3


,N,window,start,end,train_high_vol_rate,balanced_accuracy,roc_auc,f1,recall_high_vol
0,20,1,2010-01-11,2010-05-27,0.550,0.730881,0.863580,0.687500,0.559322
1,20,2,2015-04-24,2015-09-09,0.250,0.770566,0.875568,0.777778,0.711864
2,20,3,2020-08-05,2020-12-18,0.500,0.826168,0.870194,0.854701,0.847458
3,50,1,2010-01-11,2010-12-30,0.600,0.763125,0.899959,0.785714,0.745763
4,50,2,2015-01-06,2015-12-24,0.320,0.786895,0.887557,0.850394,0.915254
5,50,3,2019-12-31,2020-12-18,0.620,0.752584,0.881356,0.732673,0.627119
6,100,1,2010-01-11,2011-12-27,0.600,0.773253,0.894585,0.752475,0.644068
7,100,2,2014-07-09,2016-06-24,0.320,0.813973,0.904093,0.847458,0.847458
8,100,3,2019-01-03,2020-12-18,0.410,0.788549,0.895825,0.817391,0.796610
9,200,1,2010-01-11,2013-12-23,0.410,0.806532,0.898718,0.852459,0.881356


## 11. Aggregate classical small-data results

In [16]:
classical_summary_df = (
    classical_small_results_df
    .groupby("N")
    .agg(
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        recall_mean=("recall_high_vol", "mean"),
        recall_std=("recall_high_vol", "std")
    )
    .reset_index()
)

classical_summary_df

,N,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std,recall_mean,recall_std
0,20,0.775872,0.047865,0.869781,0.006005,0.773326,0.083689,0.706215,0.144151
1,50,0.767535,0.017576,0.889624,0.009472,0.789594,0.058956,0.762712,0.144814
2,100,0.791925,0.020569,0.898167,0.005169,0.805775,0.048545,0.762712,0.105847
3,200,0.794681,0.014565,0.894860,0.014045,0.850291,0.008156,0.898305,0.016949


In [17]:
classical_small_results_df.to_csv(
    RESULTS_DIR / "classical_small_data_windows.csv",
    index=False
)

classical_summary_df.to_csv(
    RESULTS_DIR / "classical_small_data_summary.csv",
    index=False
)

print("Classical results saved.")

Classical results saved.


## 12. Load and clean additional Bloomberg macro features

In [19]:
# --- 2Y / 10Y yields and spread ---

rates_2_10 = pd.read_csv(
    RAW_DIR / "spread 2-10.csv"
)

rates_2_10["Date"] = pd.to_datetime(
    rates_2_10["Date"]
)

rates_2_10 = rates_2_10.rename(columns={
    "USGG2YR Index": "yield_2y",
    "USGG10YR Index": "yield_10y",
    "USGG2YR Index - USGG10YR Index": "spread_2y_10y_bps"
})

rates_2_10 = (
    rates_2_10
    .set_index("Date")
    .sort_index()
)

print("2Y/10Y:")
print(rates_2_10.head())
print("Shape:", rates_2_10.shape)

2Y/10Y:
            yield_2y  yield_10y  spread_2y_10y_bps
Date                                              
2010-01-04    1.0638     3.8155          -275.1651
2010-01-05    1.0080     3.7608          -275.2852
2010-01-06    0.9920     3.8215          -282.9472
2010-01-07    1.0240     3.8235          -279.9551
2010-01-08    0.9759     3.8297          -285.3837
Shape: (3749, 3)


In [20]:
# --- VIX ---

vix = pd.read_csv(
    RAW_DIR / "VIX.csv",
    skiprows=1
)

vix.columns = ["Date", "vix"]

vix["Date"] = pd.to_datetime(
    vix["Date"],
    errors="coerce"
)

vix["vix"] = pd.to_numeric(
    vix["vix"],
    errors="coerce"
)

vix = (
    vix
    .dropna(subset=["Date"])
    .set_index("Date")
    .sort_index()
)

print("VIX:")
print(vix.head())
print("Shape:", vix.shape)
print("Range:", vix.index.min(), "to", vix.index.max())
print("Missing:", vix.isna().sum())

VIX:
              vix
Date             
2010-01-04  20.04
2010-01-05  19.35
2010-01-06  19.16
2010-01-07  19.06
2010-01-08  18.13
Shape: (3793, 1)
Range: 2010-01-04 00:00:00 to 2024-12-31 00:00:00
Missing: vix    0
dtype: int64


In [21]:
# --- 3-month Treasury yield ---

rate_3m = pd.read_csv(
    RAW_DIR / "3M-2024-2010.csv",
    skiprows=5
)

rate_3m = rate_3m.rename(columns={
    "PX_LAST": "yield_3m"
})

rate_3m["Date"] = pd.to_datetime(
    rate_3m["Date"],
    errors="coerce"
)

rate_3m["yield_3m"] = pd.to_numeric(
    rate_3m["yield_3m"],
    errors="coerce"
)

rate_3m = (
    rate_3m[
        ["Date", "yield_3m"]
    ]
    .dropna(subset=["Date"])
    .set_index("Date")
    .sort_index()
)

print("3M Treasury:")
print(rate_3m.head())
print("Shape:", rate_3m.shape)
print("Range:", rate_3m.index.min(), "to", rate_3m.index.max())
print("Missing:", rate_3m.isna().sum())

3M Treasury:
            yield_3m
Date                
2009-12-31    0.0477
2010-01-01    0.0558
2010-01-04    0.0548
2010-01-05    0.0608
2010-01-06    0.0487
Shape: (3914, 1)
Range: 2009-12-31 00:00:00 to 2024-12-31 00:00:00
Missing: yield_3m    0
dtype: int64


In [22]:
print("SPX dates:", len(data.index))
print("VIX dates:", len(vix.index))
print("3M dates:", len(rate_3m.index))
print("2Y/10Y dates:", len(rates_2_10.index))

print("\nSPX dates missing in VIX:",
      len(data.index.difference(vix.index)))

print("SPX dates missing in 3M:",
      len(data.index.difference(rate_3m.index)))

print("SPX dates missing in 2Y/10Y:",
      len(data.index.difference(rates_2_10.index)))

SPX dates: 3774
VIX dates: 3793
3M dates: 3914
2Y/10Y dates: 3749

SPX dates missing in VIX: 0
SPX dates missing in 3M: 0
SPX dates missing in 2Y/10Y: 28


In [23]:
data_extended = (
    data
    .join(vix, how="left")
    .join(rate_3m, how="left")
    .join(
        rates_2_10[
            ["yield_2y", "yield_10y", "spread_2y_10y_bps"]
        ],
        how="left"
    )
)

print("Extended shape:", data_extended.shape)

print("\nMissing values before fill:")
print(
    data_extended[
        [
            "vix",
            "yield_3m",
            "yield_2y",
            "yield_10y",
            "spread_2y_10y_bps"
        ]
    ].isna().sum()
)

Extended shape: (3774, 18)

Missing values before fill:
vix                   0
yield_3m              0
yield_2y             28
yield_10y            28
spread_2y_10y_bps    28
dtype: int64


In [24]:
rate_cols = [
    "yield_2y",
    "yield_10y",
    "spread_2y_10y_bps"
]

data_extended[rate_cols] = (
    data_extended[rate_cols]
    .ffill()
)

print("\nMissing values after forward-fill:")
print(
    data_extended[
        [
            "vix",
            "yield_3m",
            "yield_2y",
            "yield_10y",
            "spread_2y_10y_bps"
        ]
    ].isna().sum()
)


Missing values after forward-fill:
vix                  0
yield_3m             0
yield_2y             0
yield_10y            0
spread_2y_10y_bps    0
dtype: int64


In [25]:
core_feature_cols = [
    "return_1d",
    "momentum_5d",
    "gk_vol_5d",
    "gk_vol_20d",
    "vix"
]

extended_feature_cols = [
    "return_1d",
    "momentum_5d",
    "gk_vol_5d",
    "gk_vol_20d",
    "vix",
    "yield_3m",
    "spread_2y_10y_bps"
]

In [26]:
model_data_extended = data_extended[
    extended_feature_cols
    + ["future_gk_vol_5d", "high_vol"]
].dropna().copy()

train_ext = model_data_extended.loc[
    "2010-01-01":"2020-12-31"
].iloc[:-H].copy()

val_ext = model_data_extended.loc[
    "2021-01-01":"2022-12-31"
].iloc[:-H].copy()

test_ext = model_data_extended.loc[
    "2023-01-01":"2024-12-31"
].copy()

print("Train:", train_ext.shape)
print("Validation:", val_ext.shape)
print("Test:", test_ext.shape)

Train: (2759, 9)
Validation: (498, 9)
Test: (497, 9)


In [27]:
def evaluate_feature_set(feature_set, name):

    X_train = train_ext[feature_set]
    y_train = train_ext["high_vol"].astype(int)

    X_val = val_ext[feature_set]
    y_val = val_ext["high_vol"].astype(int)

    scaler = MinMaxScaler(
        feature_range=(0, np.pi),
        clip=True
    )

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = SVC(
        kernel="rbf",
        C=1,
        gamma="scale",
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_val_scaled)
    y_score = model.decision_function(X_val_scaled)

    return {
        "model": name,
        "balanced_accuracy": balanced_accuracy_score(
            y_val, y_pred
        ),
        "roc_auc": roc_auc_score(
            y_val, y_score
        ),
        "f1": f1_score(
            y_val, y_pred
        ),
        "recall": recall_score(
            y_val, y_pred
        )
    }

feature_comparison = pd.DataFrame([
    evaluate_feature_set(
        core_feature_cols,
        "Core 5 features"
    ),
    evaluate_feature_set(
        extended_feature_cols,
        "Extended 7 features"
    )
])

feature_comparison

,model,balanced_accuracy,roc_auc,f1,recall
0,Core 5 features,0.773313,0.887004,0.846986,0.951389
1,Extended 7 features,0.760466,0.853687,0.825040,0.892361


In [28]:
base_4_features = [
    "return_1d",
    "momentum_5d",
    "gk_vol_5d",
    "gk_vol_20d"
]

core_5_features = [
    "return_1d",
    "momentum_5d",
    "gk_vol_5d",
    "gk_vol_20d",
    "vix"
]

feature_comparison_4_vs_5 = pd.DataFrame([
    evaluate_feature_set(
        base_4_features,
        "Base 4 features"
    ),
    evaluate_feature_set(
        core_5_features,
        "Core 5 features + VIX"
    )
])

feature_comparison_4_vs_5

,model,balanced_accuracy,roc_auc,f1,recall
0,Base 4 features,0.795734,0.890823,0.842454,0.881944
1,Core 5 features + VIX,0.773313,0.887004,0.846986,0.951389
